# Sequence Place Recognition Benchmark Report

This report benchmarks the impact of sequence length (window size) across multiple maps. It shows per-map metrics over window sizes and overlays the cross-map mean. Aggregated mean and weighted-mean plots are also provided.


## Configuration

Paths, constants, and controls used throughout the report.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Data paths
DB_INDEX_DIR = Path(
    "/home/docker_mmpr/multimodal-place-recognition/data/2025-03-26-mmpr-datasets/keyframe-lidar-maps/mmpr_dataset_processed/map1/keyframe_map"
)
ROOT_DATA_DIR = Path(
    "/home/docker_mmpr/multimodal-place-recognition/data/2025-03-26-mmpr-datasets/keyframe-lidar-maps/mmpr_dataset_processed"
)
WEIGHTS_PATH = Path(
    "/home/docker_mmpr/multimodal-place-recognition/minkloc3d_nclt.pth"
)

assert DB_INDEX_DIR.exists(), f"Path {DB_INDEX_DIR} does not exist"
assert ROOT_DATA_DIR.exists(), f"Path {ROOT_DATA_DIR} does not exist"
assert WEIGHTS_PATH.exists(), f"Path {WEIGHTS_PATH} does not exist"

# Experiments root
EXP_ROOT = Path("/home/docker_mmpr/multimodal-place-recognition/experiments/seq_benchmarks")
EXP_ROOT.mkdir(parents=True, exist_ok=True)

# Maps to evaluate
MAPS = [f"map{i}" for i in range(2, 9)]

# Device and hyperparameters
DEVICE = "cuda"
PER_FRAME_K = 100
FINAL_K = 25
PR_PC_QUANTIZATION_SIZE = 0.05
RECALL_THRESHOLD_M = 3.0

# Controls
FORCE_RERUN = True
SKIP_IF_EXISTS = False

# Sweep settings
SEQ_LENGTHS = list(range(1, 101))


## Helpers

Utility functions to build PR caches and run the sequence benchmark sweep.


In [ ]:
from __future__ import annotations
from typing import Iterable
import json
from pathlib import Path
from IPython.display import display
from tqdm import tqdm
ing import Sequence

from mmpr.pr_infer import PRInferConfig, PRInferencer
from mmpr.seq_pr_benchmark import SequenceBenchmarkConfig, SequencePRBenchmarker


def build_pr_cache_for_map(map_name: str) -> Path:
    """Build or load PR cache for a given map.

    Args:
        map_name: Map identifier such as "map2".
    Returns:
        Path to NPZ cache file.
    """
    pr_cache_path = Path(
        f"/home/docker_mmpr/multimodal-place-recognition/experiments/pr_cache_{map_name}.npz"
    )
    if pr_cache_path.exists() and not FORCE_RERUN:
        print(f"Using existing PR cache: {pr_cache_path}")
        return pr_cache_path

    cfg_inf = PRInferConfig(
        root_data_dir=ROOT_DATA_DIR,
        map_name=map_name,
        db_map_dir=DB_INDEX_DIR,
        index_dir=DB_INDEX_DIR,
        weights=WEIGHTS_PATH,
        device=DEVICE,
        per_frame_k=PER_FRAME_K,
        pr_quant_size=PR_PC_QUANTIZATION_SIZE,
    )
    PRInferencer(cfg_inf).save(pr_cache_path)
    print(f"Built PR cache: {pr_cache_path}")
    return pr_cache_path


def _configs_match_json(d: dict, cfg: SequenceBenchmarkConfig) -> bool:
    """Return True if saved metrics.json config matches the benchmark config."""
    conf = d.get("config", {})
    try:
        return (
            int(conf.get("max_window", -1)) == int(cfg.max_window)
            and int(conf.get("per_frame_k_used", -1)) == int(cfg.per_frame_k_used)
            and int(conf.get("final_k", -1)) == int(cfg.final_k)
            and str(conf.get("recency_weighting", "")) == str(cfg.recency_weighting)
            and abs(float(conf.get("recall_threshold_m", -1.0)) - float(cfg.recall_threshold_m)) < 1e-9
        )
    except Exception:
        return False


def _row_from_metrics_json(d: dict, W: int, map_name: str) -> dict:
    """Convert metrics.json payload to a single summary row."""
    rk = d.get("recall_at_k", {}) or {}
    return {
        "w": int(W),
        "auc_pr": float(d.get("auc_pr", 0.0)),
        "f1_max": float(d.get("f1_max", 0.0)),
        "recall_at_1": float(rk.get("1", 0.0)),
        "recall_at_5": float(rk.get("5", 0.0)),
        "recall_at_10": float(rk.get("10", 0.0)),
        "recall_at_25": float(rk.get("25", 0.0)),
        "num_valid": int(d.get("num_queries_valid", 0)),
        "num_total": int(d.get("num_queries_total", 0)),
        "query_track": map_name,
    }


def run_sweep(
    map_name: str,
    pr_cache_path: Path,
    seq_lengths: Iterable[int] = SEQ_LENGTHS,
) -> pd.DataFrame:
    """Run or reuse sequence benchmark for a map across sequence lengths."""
    all_rows: list[dict] = []
    for W in tqdm(list(seq_lengths)):
        out_dir = EXP_ROOT / f"{map_name}_w{W:03d}"
        out_dir.mkdir(parents=True, exist_ok=True)
        cfg_b = SequenceBenchmarkConfig(
            db_index_dir=DB_INDEX_DIR,
            cache_path=pr_cache_path,
            root_data_dir=ROOT_DATA_DIR,
            map_name=map_name,
            max_window=int(W),
            per_frame_k_used=PER_FRAME_K,
            final_k=FINAL_K,
            recency_weighting="none",
            recall_threshold_m=RECALL_THRESHOLD_M,
        )
        metrics_path = out_dir / "metrics.json"

        if SKIP_IF_EXISTS and metrics_path.exists() and not FORCE_RERUN:
            try:
                d = json.loads(metrics_path.read_text())
                if _configs_match_json(d, cfg_b):
                    all_rows.append(_row_from_metrics_json(d, W, map_name))
                    continue
            except Exception:
                pass

        bench = SequencePRBenchmarker(cfg_b)
        artifacts = bench.run()
        bench.save(artifacts, out_dir)
        all_rows.append({
            "w": int(W),
            "auc_pr": float(artifacts.auc_pr),
            "f1_max": float(artifacts.f1_max),
            "recall_at_1": float(artifacts.recall_at_k.get(1, 0.0)),
            "recall_at_5": float(artifacts.recall_at_k.get(5, 0.0)),
            "recall_at_10": float(artifacts.recall_at_k.get(10, 0.0)),
            "recall_at_25": float(artifacts.recall_at_k.get(25, 0.0)),
            "num_valid": int(artifacts.num_queries_valid),
            "num_total": int(artifacts.num_queries_total),
            "query_track": map_name,
        })

    df = pd.DataFrame(all_rows).sort_values("w").reset_index(drop=True)
    return df


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


2025-11-11 19:57:48.370 | WARNING  | opr.models.place_recognition.pointmamba:<module>:16 - The 'pointmamba' package is not installed. Please install it manually if neccessary.


## Run Benchmarks

Build or reuse caches, run sweeps for each map, and store per-map and combined summaries.


In [3]:
# Run benchmarks for configured maps; save per-map and combined summaries
summaries: dict[str, pd.DataFrame] = {}

for m in MAPS:
    print(f"=== {m} ===")
    cache_path = build_pr_cache_for_map(m)
    df_m = run_sweep(m, cache_path, seq_lengths=SEQ_LENGTHS)
    summaries[m] = df_m
    # Save per-map summary
    out_map_dir = EXP_ROOT / m
    out_map_dir.mkdir(parents=True, exist_ok=True)
    (out_map_dir / "summary.csv").write_text(df_m.to_csv(index=False))

# Combined summary across maps
summary_all = pd.concat(list(summaries.values()), ignore_index=True)
(EXP_ROOT / "summary_all.csv").write_text(summary_all.to_csv(index=False))

display(summary_all.head(3))
display(summary_all.tail(3))


=== map2 ===
Built PR cache: /home/docker_mmpr/multimodal-place-recognition/experiments/pr_cache_map2.npz


100%|██████████| 100/100 [00:38<00:00,  2.59it/s]


=== map3 ===
Built PR cache: /home/docker_mmpr/multimodal-place-recognition/experiments/pr_cache_map3.npz


100%|██████████| 100/100 [00:38<00:00,  2.60it/s]


=== map4 ===
Built PR cache: /home/docker_mmpr/multimodal-place-recognition/experiments/pr_cache_map4.npz


...

100%|██████████| 100/100 [01:43<00:00,  1.03s/it]


=== map8 ===
Built PR cache: /home/docker_mmpr/multimodal-place-recognition/experiments/pr_cache_map8.npz


100%|██████████| 100/100 [01:16<00:00,  1.31it/s]


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.508132,0.525837,0.528517,0.785171,0.836502,0.853612,526,606,map2
1,2,0.530866,0.543863,0.538023,0.792776,0.830798,0.853612,526,606,map2
2,3,0.547476,0.557603,0.560837,0.804183,0.834601,0.851711,526,606,map2


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
697,98,0.303402,0.388688,0.403433,0.721030,0.830901,0.903004,1165,1189,map8
698,99,0.298979,0.387391,0.403433,0.718455,0.830043,0.903004,1165,1189,map8
699,100,0.297222,0.386102,0.403433,0.715880,0.829185,0.903004,1165,1189,map8


## Visualization helpers

In [4]:
def plot_metrics_vs_window_with_stats(
    summary_df,
    summary_all,
    metrics: Sequence[str] = ("auc_pr", "f1_max", "recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"),
):
    """Plot per-map metrics vs w and overlay cross-map mean and weighted mean.

    Args:
        summary_df: DataFrame for a single map (has columns 'w' and metrics).
        summary_all: Concatenated DataFrame across maps with columns 'w', 'query_track', 'num_valid', and metrics.
        metrics: metric names to visualize.
    Returns:
        dict metric -> plotly figure
    """
    figs = {}
    df = summary_df.sort_values("w").reset_index(drop=True)
    map_name = df["query_track"].iloc[0]
    # precompute simple mean once
    group = summary_all.groupby("w", as_index=False)
    mean_by_w = group[[m for m in metrics if m in summary_all.columns]].mean()

    for m in metrics:
        if m not in df.columns:
            continue
        fig = px.line(df, x="w", y=m, title=f"{map_name}: {m} vs sequence length (w)", markers=True)
        fig.update_layout(xaxis_title="sequence length (max_window)", yaxis_title=m)

        # Highlight maximum point on per-map line
        try:
            idx_max = df[m].astype(float).idxmax()
            w_star = int(df.loc[idx_max, "w"])  # sequence length at max
            y_star = float(df.loc[idx_max, m])
            fig.add_trace(
                go.Scatter(x=[w_star], y=[y_star], mode="markers", marker=dict(color="red", size=10), name="max", showlegend=False)
            )
            try:
                fig.add_vline(x=w_star, line_dash="dash", line_color="red")
            except Exception:
                fig.add_shape(type="line", x0=w_star, x1=w_star, y0=min(df[m].astype(float)), y1=max(df[m].astype(float)), line=dict(color="red", dash="dash"))
            fig.add_annotation(x=w_star, y=y_star, text=f"w={w_star}, {m}={y_star:.4f}", showarrow=True, arrowhead=2, ax=40, ay=-40)
        except Exception:
            pass

        # Overlay simple mean across maps
        if m in mean_by_w.columns:
            fig.add_trace(
                go.Scatter(
                    x=mean_by_w["w"],
                    y=mean_by_w[m].astype(float),
                    mode="lines",
                    name="mean",
                    line=dict(color="green", dash="dash"),
                    showlegend=True,
                )
            )

        # Overlay weighted mean across maps (weights = num_valid per map)
        try:
            wmean_series = (
                summary_all
                .groupby("w")
                .apply(lambda g: float(np.average(g[m].astype(float), weights=g["num_valid"].astype(float))), include_groups=False)
                .reset_index(name=m)
            )
            fig.add_trace(
                go.Scatter(
                    x=wmean_series["w"],
                    y=wmean_series[m].astype(float),
                    mode="lines",
                    name="weighted mean",
                    line=dict(color="purple", dash="dot"),
                    showlegend=True,
                )
            )
        except Exception:
            pass

        figs[m] = fig
        fig.show()
    return figs


In [5]:
def plot_aggregate_mean_median(
    summary_all,
    metrics: Sequence[str] = ("auc_pr", "f1_max", "recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"),
):
    """Draw separate figures that show only mean and weighted mean across maps, with maxima highlighted.

    Args:
        summary_all: Concatenated DataFrame across maps with columns 'w', 'query_track', 'num_valid', and metrics.
        metrics: metric names to visualize.
    Returns:
        dict metric -> plotly figure
    """
    figs = {}
    group = summary_all.groupby("w", as_index=False)
    mean_by_w = group[[m for m in metrics if m in summary_all.columns]].mean()
    # Weighted mean by num_valid
    def _weighted_series(g):
        return pd.Series({k: float(np.average(g[k].astype(float), weights=g["num_valid"].astype(float))) for k in metrics if k in g.columns})
    wmean_by_w = summary_all.groupby("w").apply(_weighted_series, include_groups=False).reset_index()

    for m in metrics:
        if m not in summary_all.columns:
            continue
        fig = go.Figure()
        # Mean line
        fig.add_trace(
            go.Scatter(
                x=mean_by_w["w"],
                y=mean_by_w[m].astype(float),
                mode="lines",
                name="mean",
                line=dict(color="green", dash="dash"),
                showlegend=True,
            )
        )
        # Weighted mean line
        if m in wmean_by_w.columns:
            fig.add_trace(
                go.Scatter(
                    x=wmean_by_w["w"],
                    y=wmean_by_w[m].astype(float),
                    mode="lines",
                    name="weighted mean",
                    line=dict(color="purple", dash="dot"),
                    showlegend=True,
                )
            )

        # Maxima on mean
        try:
            idx_mean_max = mean_by_w[m].astype(float).idxmax()
            w_mean_max = int(mean_by_w.loc[idx_mean_max, "w"])
            y_mean_max = float(mean_by_w.loc[idx_mean_max, m])
            fig.add_trace(
                go.Scatter(
                    x=[w_mean_max],
                    y=[y_mean_max],
                    mode="markers",
                    marker=dict(color="green", size=9, symbol="diamond"),
                    name="mean max",
                    showlegend=False,
                )
            )
            try:
                fig.add_vline(x=w_mean_max, line_dash="dash", line_color="green")
            except Exception:
                fig.add_shape(
                    type="line",
                    x0=w_mean_max,
                    x1=w_mean_max,
                    y0=min(mean_by_w[m].astype(float)),
                    y1=max(mean_by_w[m].astype(float)),
                    line=dict(color="green", dash="dash"),
                )
            fig.add_annotation(
                x=w_mean_max,
                y=y_mean_max,
                text=f"mean max: w={w_mean_max}, {m}={y_mean_max:.4f}",
                showarrow=True,
                arrowhead=2,
                ax=40,
                ay=-40,
            )
        except Exception:
            pass

        # Maxima on weighted mean
        try:
            idx_wmean_max = wmean_by_w[m].astype(float).idxmax()
            w_wmean_max = int(wmean_by_w.loc[idx_wmean_max, "w"])
            y_wmean_max = float(wmean_by_w.loc[idx_wmean_max, m])
            fig.add_trace(
                go.Scatter(
                    x=[w_wmean_max],
                    y=[y_wmean_max],
                    mode="markers",
                    marker=dict(color="purple", size=9, symbol="x"),
                    name="weighted mean max",
                    showlegend=False,
                )
            )
            try:
                fig.add_vline(x=w_wmean_max, line_dash="dot", line_color="purple")
            except Exception:
                fig.add_shape(
                    type="line",
                    x0=w_wmean_max,
                    x1=w_wmean_max,
                    y0=min(wmean_by_w[m].astype(float)),
                    y1=max(wmean_by_w[m].astype(float)),
                    line=dict(color="purple", dash="dot"),
                )
            fig.add_annotation(
                x=w_wmean_max,
                y=y_wmean_max,
                text=f"w-mean max: w={w_wmean_max}, {m}={y_wmean_max:.4f}",
                showarrow=True,
                arrowhead=2,
                ax=40,
                ay=-40,
            )
        except Exception:
            pass

        fig.update_layout(
            title=f"{m} (mean/weighted mean across maps) vs sequence length (w)",
            xaxis_title="sequence length (max_window)",
            yaxis_title=m,
        )
        figs[m] = fig
        fig.show()
    return figs


# Results

## Per-map values

In [6]:
# Compute and display aggregated stats, and draw overlays on plots for each map
metrics_cols = ("auc_pr", "f1_max", "recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25")
summary_mean_by_w = summary_all.groupby("w")[list(metrics_cols)].mean().reset_index()
summary_weighted_mean_by_w = (
    summary_all
    .groupby("w")
    .apply(lambda g: pd.Series({k: float(np.average(g[k].astype(float), weights=g["num_valid"].astype(float))) for k in metrics_cols}), include_groups=False)
    .reset_index()
)

# Render plots, overlaying mean and weighted mean across all maps
for mname, df_map in summaries.items():
    print(f"\n=== {mname}: mean and weighted-mean overlays ===")
    display(df_map.head(30))
    plot_metrics_vs_window_with_stats(df_map, summary_all)



=== map2: mean and weighted-mean overlays ===


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.508132,0.525837,0.528517,0.785171,0.836502,0.853612,526,606,map2
1,2,0.530866,0.543863,0.538023,0.792776,0.830798,0.853612,526,606,map2
2,3,0.547476,0.557603,0.560837,0.804183,0.834601,0.851711,526,606,map2
3,4,0.560151,0.571952,0.576046,0.815589,0.840304,0.851711,526,606,map2
4,5,0.570855,0.581755,0.583650,0.817490,0.844106,0.851711,526,606,map2
5,6,0.580132,0.589663,0.596958,0.821293,0.847909,0.853612,526,606,map2
6,7,0.588087,0.594115,0.602662,0.823194,0.849810,0.855513,526,606,map2
7,8,0.593274,0.594920,0.608365,0.823194,0.851711,0.855513,526,606,map2
8,9,0.598337,0.596057,0.614068,0.821293,0.851711,0.855513,526,606,map2
9,10,0.602796,0.595911,0.619772,0.821293,0.853612,0.855513,526,606,map2



=== map3: mean and weighted-mean overlays ===


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.491822,0.571656,0.777372,0.972628,0.990876,0.996350,548,629,map3
1,2,0.511916,0.583884,0.813869,0.978102,0.990876,0.994526,548,629,map3
2,3,0.525148,0.592344,0.822993,0.981752,0.992701,0.994526,548,629,map3
3,4,0.535479,0.599303,0.832117,0.985401,0.992701,0.994526,548,629,map3
4,5,0.543674,0.606571,0.835766,0.987226,0.992701,0.994526,548,629,map3
5,6,0.550420,0.612341,0.844891,0.985401,0.992701,0.994526,548,629,map3
6,7,0.556489,0.617334,0.850365,0.983577,0.990876,0.994526,548,629,map3
7,8,0.561277,0.622902,0.857664,0.983577,0.990876,0.994526,548,629,map3
8,9,0.566084,0.627783,0.861314,0.983577,0.990876,0.994526,548,629,map3
9,10,0.569725,0.632099,0.864964,0.979927,0.989051,0.994526,548,629,map3



=== map4: mean and weighted-mean overlays ===


...

,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.295731,0.329133,0.400312,0.660436,0.755452,0.898754,642,642,map6
1,2,0.313326,0.338920,0.415888,0.661994,0.761682,0.903427,642,642,map6
2,3,0.325671,0.346432,0.431464,0.672897,0.763240,0.908100,642,642,map6
3,4,0.335001,0.353323,0.450156,0.679128,0.769470,0.908100,642,642,map6
4,5,0.344263,0.361380,0.468847,0.680685,0.777259,0.915888,642,642,map6
5,6,0.352017,0.367236,0.481308,0.685358,0.785047,0.917445,642,642,map6
6,7,0.358645,0.370792,0.493769,0.691589,0.789720,0.919003,642,642,map6
7,8,0.365092,0.372636,0.501558,0.702492,0.788162,0.917445,642,642,map6
8,9,0.370658,0.375064,0.506231,0.711838,0.791277,0.920561,642,642,map6
9,10,0.375652,0.383319,0.506231,0.714953,0.792835,0.920561,642,642,map6



=== map7: mean and weighted-mean overlays ===


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.337614,0.444439,0.480106,0.765915,0.846817,0.911804,1508,1596,map7
1,2,0.352964,0.454481,0.507294,0.778515,0.853448,0.911141,1508,1596,map7
2,3,0.364593,0.460635,0.523210,0.783156,0.856764,0.915782,1508,1596,map7
3,4,0.373110,0.465266,0.539788,0.793103,0.861406,0.917109,1508,1596,map7
4,5,0.379013,0.466650,0.543767,0.797745,0.867374,0.918435,1508,1596,map7
5,6,0.385062,0.469212,0.549072,0.795093,0.872016,0.921088,1508,1596,map7
6,7,0.391202,0.472509,0.554377,0.798408,0.874668,0.923740,1508,1596,map7
7,8,0.396434,0.475479,0.561008,0.800398,0.872679,0.924403,1508,1596,map7
8,9,0.400257,0.477778,0.564987,0.801724,0.872016,0.924403,1508,1596,map7
9,10,0.403766,0.480016,0.568302,0.802387,0.872679,0.924403,1508,1596,map7



=== map8: mean and weighted-mean overlays ===


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.409960,0.445167,0.365665,0.643777,0.763948,0.885837,1165,1189,map8
1,2,0.435262,0.463267,0.375966,0.663519,0.772532,0.890129,1165,1189,map8
2,3,0.453544,0.476881,0.395708,0.665236,0.775966,0.893562,1165,1189,map8
3,4,0.467893,0.486607,0.406867,0.669528,0.777682,0.900429,1165,1189,map8
4,5,0.478925,0.491386,0.425751,0.678970,0.782833,0.903004,1165,1189,map8
5,6,0.488326,0.494646,0.440343,0.684979,0.783691,0.906438,1165,1189,map8
6,7,0.497391,0.497122,0.454077,0.690129,0.787983,0.908155,1165,1189,map8
7,8,0.504840,0.497958,0.466094,0.693562,0.791416,0.911588,1165,1189,map8
8,9,0.511302,0.500105,0.470386,0.695279,0.792275,0.913305,1165,1189,map8
9,10,0.516768,0.503181,0.476395,0.696996,0.792275,0.915880,1165,1189,map8


## Aggregated values

In [7]:
# Display aggregated tables (mean and weighted-mean)
display(summary_mean_by_w.head(30))
try:
    display(summary_weighted_mean_by_w.head(30))
except Exception:
    pass

# Aggregate-only plots (mean and weighted mean across maps) with maxima
plot_aggregate_mean_median(summary_all);


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25
0,1,0.400179,0.461160,0.512703,0.774362,0.851290,0.918553
1,2,0.419315,0.474950,0.532632,0.784522,0.854343,0.920544
2,3,0.432687,0.484145,0.547614,0.790381,0.859096,0.922093
3,4,0.443102,0.491851,0.562386,0.798176,0.863712,0.924287
4,5,0.451370,0.497647,0.574754,0.800986,0.867599,0.926016
5,6,0.458653,0.502424,0.585023,0.803834,0.869456,0.928139
6,7,0.465385,0.506098,0.593342,0.807802,0.872208,0.929900
7,8,0.470972,0.508956,0.599078,0.811714,0.873545,0.931549
8,9,0.475778,0.511574,0.603623,0.813929,0.875041,0.933101
9,10,0.479952,0.514727,0.606950,0.815611,0.875969,0.934010


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25
0,1,0.380961,0.451071,0.487603,0.756451,0.839939,0.913813
1,2,0.399651,0.464421,0.507337,0.768258,0.844325,0.916006
2,3,0.412819,0.473274,0.522516,0.773486,0.849047,0.918199
3,4,0.423035,0.480386,0.537359,0.781245,0.853601,0.920897
4,5,0.430986,0.485260,0.549502,0.784955,0.857986,0.922584
5,6,0.438081,0.489379,0.559285,0.787317,0.859841,0.924945
6,7,0.444796,0.492795,0.567718,0.791364,0.862877,0.926969
7,8,0.450460,0.495544,0.573959,0.795075,0.864058,0.928993
8,9,0.455201,0.498063,0.578344,0.797099,0.865407,0.930511
9,10,0.459363,0.501088,0.581717,0.798786,0.866251,0.931523
